# 002 · Mocking with unittest.mock

> **Standard Library › Testing & Debugging** · Python 3.14

`unittest.mock` lets you replace real collaborators (databases, HTTP clients, clocks, environment variables) with programmable fakes and then check how they were used. Unlike .NET, where mocking needs a NuGet package such as Moq or NSubstitute, it ships with Python — and because Python is dynamic, it can even replace module-level functions, which no .NET mocking library can do without special tooling.

**In this notebook you will learn:**

- how `Mock` / `MagicMock` map onto Moq's `Setup`, `Returns`, `Throws` and `Verify`
- how to inspect calls with `call_args`, `call_count` and `mock_calls`
- how `side_effect` raises exceptions, returns sequences or runs a function
- why `spec` / `autospec` matter (they catch typos and wrong signatures)
- how to use `patch`, `patch.object` and `patch.dict` — and the "patch where it is looked up" rule
- `AsyncMock` for coroutines, and dependency injection as the alternative to patching

## .NET mental model

| Moq (C#) | `unittest.mock` (Python) |
|---|---|
| `var mock = new Mock<IRepo>();` | `repo = Mock()` or, better, `create_autospec(Repo, instance=True)` |
| `mock.Object` | the mock *is* the object — pass `repo` directly |
| `mock.Setup(r => r.Get(1)).Returns(order)` | `repo.get.return_value = order` (for **any** arguments) |
| `.Throws<NotFoundException>()` | `repo.get.side_effect = NotFound()` |
| `SetupSequence(...).Returns(1).Returns(2)` | `repo.get.side_effect = [1, 2]` |
| `.Returns((int id) => ...)` / `.Callback(...)` | `repo.get.side_effect = lambda id: ...` |
| `.ReturnsAsync(order)` | `AsyncMock(return_value=order)` |
| `mock.Verify(r => r.Save(order), Times.Once)` | `repo.save.assert_called_once_with(order)` |
| `Times.Never` | `repo.save.assert_not_called()` |
| `It.IsAny<T>()` | `unittest.mock.ANY` |
| `MockBehavior.Strict` | closest: `spec_set=` / `create_autospec` (rejects unknown attributes) |
| — (needs Fakes/Shims or a wrapper interface) | `patch("module.function")` — replace *anything* at runtime |

The biggest mental shift: a Python `Mock` is **permissive by default**. Any attribute you touch springs into existence as another `Mock`, and every call succeeds and returns yet another `Mock`.

## `Mock` basics: return values and recording calls

```csharp
var repo = new Mock<IOrderRepository>();
repo.Setup(r => r.GetTotal(42)).Returns(99.5m);
var total = repo.Object.GetTotal(42);
repo.Verify(r => r.GetTotal(42), Times.Once());
```

In [ ]:
from unittest.mock import Mock, MagicMock, ANY, call

repo = Mock()
repo.get_total.return_value = 99.5      # Setup(...).Returns(99.5)

print(repo.get_total(42))               # 99.5
print(repo.get_total("anything"))       # 99.5 too — no argument matching!

repo.get_total.assert_called_with("anything")   # checks the LAST call
print(repo.get_total.call_count)        # 2

⚠️ **Gotcha:** `return_value` ignores the arguments completely. Moq's `Setup(r => r.Get(1))` only matches `1`; the Python equivalent of argument-specific behaviour is a `side_effect` function (shown below).

## Inspecting calls

Every call is recorded. `call_args` is the most recent call, `call_args_list` all calls to *that* attribute, and `mock_calls` every call to the mock **and its children**, in order. The `call` helper builds expected values to compare against.

In [ ]:
notifier = Mock()
notifier.send("alice", subject="Welcome")
notifier.send("bob", subject="Reminder")
notifier.flush()

print(notifier.send.call_count)                 # 2
print(notifier.send.call_args)                  # call('bob', subject='Reminder')
print(notifier.send.call_args.args, notifier.send.call_args.kwargs)
print(notifier.send.call_args_list)
print(notifier.mock_calls)                      # includes call.flush()

## Verifying calls (`Verify`)

| Moq | `unittest.mock` |
|---|---|
| `Verify(x => x.M(), Times.Once())` | `m.assert_called_once()` |
| `Verify(x => x.M(1, "a"), Times.Once())` | `m.assert_called_once_with(1, "a")` |
| `Verify(x => x.M(1), Times.AtLeastOnce())` | `m.assert_any_call(1)` |
| `Verify(x => x.M(), Times.Never())` | `m.assert_not_called()` |
| `Verify(x => x.M(It.IsAny<string>()))` | `m.assert_called_with(ANY)` |
| `VerifyNoOtherCalls()` + `MockSequence` | `parent.assert_has_calls([...])` / compare `mock_calls` |

A failed verification raises `AssertionError` with a helpful "Expected / Actual" message:

In [ ]:
notifier.send.assert_any_call("alice", subject=ANY)        # passes
notifier.assert_has_calls([call.send("bob", subject="Reminder"), call.flush()])
                                       # a consecutive run of calls, in this order
try:
    notifier.send.assert_called_once_with("alice", subject="Welcome")
except AssertionError as e:
    print(f"{type(e).__name__}: {e}")

`reset_mock()` clears the recorded calls (handy between "arrange" phases):

In [ ]:
notifier.reset_mock()
notifier.send.assert_not_called()
print(notifier.mock_calls)      # []

## `side_effect`: exceptions, sequences and functions

`side_effect` takes priority over `return_value` and covers three Moq features in one:

- an **exception** (class or instance) → raised when called (`.Throws(...)`)
- an **iterable** → each call returns the next item (`SetupSequence`); an exception item in the iterable is raised
- a **function** → called with the same arguments; its result is returned (`.Returns(Func<...>)` / `.Callback`)

In [ ]:
gateway = Mock()

gateway.charge.side_effect = TimeoutError("payment gateway timed out")
try:
    gateway.charge(10)
except TimeoutError as e:
    print(f"{type(e).__name__}: {e}")

gateway.charge.side_effect = [ConnectionError("flaky"), "ok-1", "ok-2"]   # retry scenario
for attempt in range(3):
    try:
        print(attempt, gateway.charge(10))
    except ConnectionError as e:
        print(attempt, f"{type(e).__name__}: {e}")

In [ ]:
prices = {"apple": 0.5, "pear": 0.75}
catalogue = Mock()
catalogue.price_of.side_effect = lambda sku: prices[sku]   # argument-specific behaviour

print(catalogue.price_of("pear"))       # 0.75
try:
    catalogue.price_of("kiwi")
except KeyError as e:
    print(f"{type(e).__name__}: {e}")

⚠️ **Gotcha:** an iterable `side_effect` raises `StopIteration` once it is exhausted — it does not fall back to `return_value`.

## `MagicMock`: dunder methods too

A plain `Mock` does not support Python protocols such as `len()`, iteration, indexing or `with`. `MagicMock` pre-configures those "magic" methods, which you can then customise. `patch` uses `MagicMock` by default.

In [ ]:
plain = Mock()
try:
    len(plain)
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

magic = MagicMock()
magic.__len__.return_value = 3
magic.__iter__.return_value = iter(["a", "b", "c"])
print(len(magic), list(magic))

with magic as entered:                       # context-manager protocol works
    print(entered is magic.__enter__.return_value)    # True

## `spec` and `autospec`: catching typos

Because a bare `Mock` accepts *anything*, a test can pass while the production code is broken — for example if the real class has no such method, or you misspell a configuration attribute. In C# the compiler protects you; in Python you opt into checking with a **spec**:

- `Mock(spec=Cls)` — only attributes that exist on `Cls` are allowed (and `isinstance` checks pass)
- `create_autospec(Cls, instance=True)` — additionally checks **call signatures** of methods, recursively
- `spec_set=Cls` — also forbids *setting* unknown attributes

In [ ]:
from unittest.mock import create_autospec

class OrderRepository:
    def get(self, order_id: int) -> dict: ...
    def save(self, order: dict) -> None: ...

loose = Mock()
loose.retrun_value = {"id": 1}             # typo! silently creates an attribute
print(isinstance(loose(), Mock))           # True — returns a Mock, not our dict
print(isinstance(loose.gett(1), Mock))     # misspelt method "works" too

In [ ]:
strict = create_autospec(OrderRepository, instance=True)

for action in (lambda: strict.gett(1),     # unknown attribute
               lambda: strict.get(),       # wrong signature (missing order_id)
               lambda: strict.save(1, 2)): # too many arguments
    try:
        action()
    except (AttributeError, TypeError) as e:
        print(f"{type(e).__name__}: {e}")

strict.get.return_value = {"id": 7}
print(strict.get(7), isinstance(strict, OrderRepository))

⚠️ **Gotcha:** mock *does* catch misspelt assertion methods for you — any unknown attribute starting with `assert`, `assret`, `asert`, `aseert`, `assrt`, or names like `called_once_with`, raises `AttributeError`. But forgetting the parentheses (`m.assert_called_once` instead of `m.assert_called_once()`) just references the method and asserts nothing. A spec or autospec is still the best protection against everything else.

In [ ]:
m = Mock()
for name in ("assert_called_once_wtih", "called_once_with"):
    try:
        getattr(m, name)
    except AttributeError as e:
        print(f"{type(e).__name__}: {str(e)[:60]}...")

m.assert_called_once     # no call -> no check, no error, test "passes"
print("still green, but nothing was asserted")

## `patch`: replacing things at runtime

`patch(target)` temporarily replaces the object named by the dotted string `target` with a `MagicMock` (or whatever you pass as `new=`), and **always restores the original** afterwards. Use it as a context manager or as a decorator; the decorator passes the mock in as an extra argument.

The closest .NET equivalents are Microsoft Fakes "shims" or Pose/Harmony — rarely used, because .NET code is usually designed for injection. In Python `patch` is everyday tooling.

In [ ]:
import os
import random
from unittest.mock import patch

def new_ticket() -> str:
    return f"TICKET-{random.randint(1000, 9999)}"

with patch("random.randint", return_value=4242) as fake_randint:
    print(new_ticket())                      # TICKET-4242
    fake_randint.assert_called_once_with(1000, 9999)

print(isinstance(random.randint, Mock))     # False — original restored

As a decorator (on a test method or any function). With several decorators, mocks are passed **bottom-up** — the decorator closest to the function gives the first argument:

In [ ]:
@patch("os.getcwd", return_value="/srv/app")
@patch("os.cpu_count", return_value=64)
def describe_host(fake_cpu_count, fake_getcwd):      # bottom decorator first!
    return f"{os.cpu_count()} cores in {os.getcwd()}"

print(describe_host())                 # 64 cores in /srv/app

## ⚠️ Gotcha: patch where it is *looked up*, not where it is *defined*

`from module import name` copies a **reference** into the importing module's namespace. Patching the original module afterwards does not change that copy. You must patch the name in the module that **uses** it.

To demonstrate with real modules, we write two tiny files into a temporary directory and put it on `sys.path`:

In [ ]:
import importlib
import sys
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()
root = Path(tmp.name)
(root / "weather.py").write_text(
    "def fetch_temperature(city):\n"
    "    raise RuntimeError('would call a real web service')\n"
)
(root / "dashboard.py").write_text(
    "from weather import fetch_temperature\n\n"
    "def headline(city):\n"
    "    return f'{city}: {fetch_temperature(city)} °C'\n"
)
sys.path.insert(0, str(root))
import dashboard
print(sorted(p.name for p in root.glob("*.py")))

In [ ]:
# WRONG: patches weather.fetch_temperature, but dashboard holds its own reference
with patch("weather.fetch_temperature", return_value=21):
    try:
        print(dashboard.headline("Oslo"))
    except RuntimeError as e:
        print(f"{type(e).__name__}: {e}")

# RIGHT: patch the name where dashboard looks it up
with patch("dashboard.fetch_temperature", return_value=21):
    print(dashboard.headline("Oslo"))        # Oslo: 21 °C

Had `dashboard.py` done `import weather` and called `weather.fetch_temperature(city)`, then patching `"weather.fetch_temperature"` *would* work — the lookup happens on the `weather` module at call time. Rule of thumb: **patch the attribute on the object your code reads it from.**

Clean up the temporary modules:

In [ ]:
sys.path.remove(str(root))
for name in ("dashboard", "weather"):
    sys.modules.pop(name, None)
tmp.cleanup()
print(root.exists())    # False

## `patch.object` and `patch.dict`

- `patch.object(target_obj, "attribute", ...)` patches an attribute on an object you already hold — no dotted string, so no "where is it looked up" confusion and rename-safe tooling. Combine with `autospec=True` to keep signature checking.
- `patch.dict(mapping, values, clear=False)` temporarily adds/overrides keys of a dict — perfect for `os.environ`. Like `Environment.SetEnvironmentVariable`, but automatically undone.

In [ ]:
class PaymentService:
    def charge(self, amount: float) -> str:
        raise RuntimeError("real network call")

    def checkout(self, amount: float) -> str:
        return f"receipt:{self.charge(amount)}"

with patch.object(PaymentService, "charge", autospec=True, return_value="tx-001") as fake:
    service = PaymentService()
    print(service.checkout(9.99))                 # receipt:tx-001
    fake.assert_called_once_with(service, 9.99)   # autospec records `self` too

In [ ]:
def api_base_url() -> str:
    return os.environ.get("SHOP_API_URL", "https://api.example.invalid")

with patch.dict(os.environ, {"SHOP_API_URL": "http://127.0.0.1:8080"}):
    print(api_base_url())          # http://127.0.0.1:8080
print(api_base_url())              # https://api.example.invalid (restored)

**Tip:** inside a `TestCase` you can also start a patcher in `setUp` and register its undo: `self.addCleanup(patcher.stop)` after `patcher.start()` — the equivalent of initialising a mock field in an NUnit `[SetUp]`.

## `AsyncMock` for coroutines

Mocking an `async` method with a plain `Mock` gives you something that is not awaitable. `AsyncMock` returns a coroutine when called; `return_value` / `side_effect` apply to the **awaited** result — like Moq's `ReturnsAsync` / `ThrowsAsync`. `create_autospec` and `patch` automatically use `AsyncMock` for `async def` functions.

In [ ]:
from unittest.mock import AsyncMock

client = Mock()
client.fetch_stock = AsyncMock(return_value=12)
client.fetch_price = AsyncMock(side_effect=[0.5, TimeoutError("slow")])

print(await client.fetch_stock("apple"))      # 12
print(await client.fetch_price("apple"))      # 0.5
try:
    await client.fetch_price("apple")
except TimeoutError as e:
    print(f"{type(e).__name__}: {e}")

client.fetch_stock.assert_awaited_once_with("apple")
print(client.fetch_price.await_count)         # 2

## The alternative: dependency injection

In C# you mock *interfaces* that are injected through constructors. That works just as well in Python — and often better than `patch`, because the test does not need to know module internals. Python's duck typing (or a `Protocol` *(see Syntax › Classes & OOP › 006 · Abstract Classes and Protocols)*) means you don't even need an interface: any object or function with the right shape will do.

```csharp
public class Greeter(TimeProvider clock) { ... }   // inject TimeProvider, fake it in tests
```

In [ ]:
from collections.abc import Callable
from datetime import datetime

class Greeter:
    def __init__(self, now: Callable[[], datetime] = datetime.now):
        self._now = now                      # injected dependency with a real default

    def greet(self, name: str) -> str:
        part = "morning" if self._now().hour < 12 else "afternoon"
        return f"Good {part}, {name}!"

# No patching required: pass a stub function (or a Mock) in the test
print(Greeter(now=lambda: datetime(2025, 1, 1, 9)).greet("alice"))
print(Greeter(now=Mock(return_value=datetime(2025, 1, 1, 15))).greet("bob"))

**Tip:** reach for `patch` when you cannot change the code's shape (third-party code, module-level functions, `os.environ`), and for injection when you design the code yourself. Over-patching produces brittle tests that break on harmless refactorings.

## Summary

| Task | C# / .NET (Moq) | Python (`unittest.mock`) |
|---|---|---|
| Create a fake | `new Mock<IRepo>()` | `Mock()`, `MagicMock()`, `create_autospec(Repo, instance=True)` |
| Return a value | `.Setup(...).Returns(x)` | `m.method.return_value = x` |
| Throw | `.Throws<T>()` | `m.method.side_effect = T(...)` |
| Sequence of results | `SetupSequence(...)` | `side_effect = [a, b, Exc()]` |
| Compute from arguments | `.Returns((args) => ...)` | `side_effect = lambda *args: ...` |
| Verify call | `Verify(..., Times.Once)` | `assert_called_once_with(...)` |
| Any argument | `It.IsAny<T>()` | `ANY` |
| Inspect calls | `mock.Invocations` | `call_args`, `call_args_list`, `call_count`, `mock_calls` |
| Strict member checking | compiler + `MockBehavior.Strict` | `spec=`, `spec_set=`, `autospec=True` |
| Async method | `.ReturnsAsync(x)` | `AsyncMock(return_value=x)`, `assert_awaited_once_with` |
| Replace static/module member | Fakes shims (rare) | `patch("where.it.is.used")`, `patch.object(obj, "attr")` |
| Fake env vars | `Environment.SetEnvironmentVariable` + restore | `patch.dict(os.environ, {...})` |

## Exercises

1. Create a `Mock` named `mailer` whose `send` method returns `True` the first two times and raises `ConnectionError` the third time. Call it three times (catching the error), then assert it was called exactly three times and that the last call was `send("carol")`.
2. Given `class Inventory: def reserve(self, sku: str, qty: int) -> bool: ...`, create an autospecced instance mock, show that `inv.reserve("apple")` raises `TypeError`, and make `inv.reserve("apple", 2)` return `True`.
3. Write `def is_debug() -> bool: return os.environ.get("APP_DEBUG") == "1"` and test both branches with `patch.dict` (hint: `clear=True` gives you an environment *without* the variable).
4. Write an `async def total_stock(client, skus)` that awaits `client.stock(sku)` for each SKU and returns the sum. Test it with an `AsyncMock` whose `side_effect` is a dict lookup, and assert it was awaited twice.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
import os
from unittest.mock import AsyncMock, Mock, call, create_autospec, patch

# 1
mailer = Mock()
mailer.send.side_effect = [True, True, ConnectionError("smtp down")]
for name in ("alice", "bob", "carol"):
    try:
        print(mailer.send(name))
    except ConnectionError as e:
        print(f"{type(e).__name__}: {e}")
assert mailer.send.call_count == 3
mailer.send.assert_called_with("carol")

# 2
class Inventory:
    def reserve(self, sku: str, qty: int) -> bool: ...

inv = create_autospec(Inventory, instance=True)
try:
    inv.reserve("apple")
except TypeError as e:
    print(f"{type(e).__name__}: {e}")
inv.reserve.return_value = True
print(inv.reserve("apple", 2))

# 3
def is_debug() -> bool:
    return os.environ.get("APP_DEBUG") == "1"

with patch.dict(os.environ, {"APP_DEBUG": "1"}):
    assert is_debug() is True
with patch.dict(os.environ, {}, clear=True):
    assert is_debug() is False
print("both branches tested")

# 4
async def total_stock(client, skus):
    return sum([await client.stock(sku) for sku in skus])

stock = {"apple": 5, "pear": 3}
client = Mock()
client.stock = AsyncMock(side_effect=lambda sku: stock[sku])
print(await total_stock(client, ["apple", "pear"]))   # 8
assert client.stock.await_count == 2
client.stock.assert_has_awaits([call("apple"), call("pear")])
```

</details>